## Common imports

In [1]:
from src.barbara_functions_for_notebooks_1_2_3_4_5 import filter_ra_only, export_to_csv
import pandas as pd
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from src.barbara_functions_for_notebooks_1_2_3_4_5 import load_autoimmune_data, get_duplicate_patient_ids, get_duplicate_values, check_duplicate_protein_ids, transpose_autoimmune_data, split_bl_m6, harmonisation_validator, create_annotation_table

## Define directory and file paths

In [2]:
PROJECT_ROOT = Path.cwd().parents[1]
DATA_DIR = PROJECT_ROOT / "datasets"
INITIAL_AUTOIMMUNE_FILE = "1_Initial_Autoimmune.xlsx"
INITIAL_AUTOIMMUNE_FILE_RA_ONLY = "2_Initial_Autoimmune_RA_Only.csv"
from pathlib import Path

print(Path.cwd())
print(Path(DATA_DIR))
print(Path(DATA_DIR) / INITIAL_AUTOIMMUNE_FILE)

C:\Users\barbs\COMPUTER SCIENCE\FH CAMPUS\6_COURSES SS 26\KI Wahlfach Projekt\Source Code_Main_1\RA-Map-EDA\notebooks\preprocessing
C:\Users\barbs\COMPUTER SCIENCE\FH CAMPUS\6_COURSES SS 26\KI Wahlfach Projekt\Source Code_Main_1\RA-Map-EDA\datasets
C:\Users\barbs\COMPUTER SCIENCE\FH CAMPUS\6_COURSES SS 26\KI Wahlfach Projekt\Source Code_Main_1\RA-Map-EDA\datasets\1_Initial_Autoimmune.xlsx


## Load the initial_autoimmune dataset

In [3]:
initial_autoimmune_dataset = load_autoimmune_data(DATA_DIR,INITIAL_AUTOIMMUNE_FILE)

Loaded: C:\Users\barbs\COMPUTER SCIENCE\FH CAMPUS\6_COURSES SS 26\KI Wahlfach Projekt\Source Code_Main_1\RA-Map-EDA\datasets\1_Initial_Autoimmune.xlsx
Shape: (163, 597)
Index name: None


## Create a feature annotation table - PatientID_GeneID and Gene Name (to have a mapping for later analysis)

In [4]:
annotation_table = create_annotation_table(
    initial_autoimmune_dataset,
    protein_id_col="ProteinID",
    gene_symbol_col="Gene Symbol",
    gene_name_col="Gene Name"
)

## Export the annotation table as .csv

In [5]:
annotation_table.to_csv(DATA_DIR / "0_Initial_Autoimmune_Annotation_Table.csv", index=False)

## Remove non-RA patients from the dataset to focus on RA-specific analysis

In [4]:
ra_only_initial_autoimmune_dataset = filter_ra_only(initial_autoimmune_dataset)

Removed 93 VAC columns
Remaining dataset shape: (163, 504)


## Export the data-frame as .csv

In [5]:
export_to_csv(ra_only_initial_autoimmune_dataset, f"{DATA_DIR}/2_Initial_Autoimmune_RA_Only.csv")

File saved to: C:\Users\barbs\COMPUTER SCIENCE\FH CAMPUS\6_COURSES SS 26\KI Wahlfach Projekt\Source Code_Main_1\RA-Map-EDA\datasets/2_Initial_Autoimmune_RA_Only.csv


## Transpose the Matrix Dataset so that ProteinId is in the columns and PatientId is in the rows

In [6]:
ra_only_initial_autoimmune_dataset_transposed = transpose_autoimmune_data(ra_only_initial_autoimmune_dataset)

Transposition complete
Shape: (500, 163)
Index preview: Index(['TAC1241_BL', 'TAC1241_M6', 'TAC1147_BL', 'TAC1147_M6', 'TAC1094_BL'], dtype='str', name='Patient_Timepoint')


## Export the data-frame as .csv

In [7]:
export_to_csv(ra_only_initial_autoimmune_dataset_transposed, f"{DATA_DIR}/3_Initial_Autoimmune_RA_Only_Transposed.csv")

File saved to: C:\Users\barbs\COMPUTER SCIENCE\FH CAMPUS\6_COURSES SS 26\KI Wahlfach Projekt\Source Code_Main_1\RA-Map-EDA\datasets/3_Initial_Autoimmune_RA_Only_Transposed.csv


## Split the transposed dataset into BL and M6

In [8]:
df_bl, df_m6 = split_bl_m6(ra_only_initial_autoimmune_dataset_transposed)

#print(df_bl.index[:5])
#print(df_m6.index[:5])

BL shape: (265, 163)
M6 shape: (235, 163)


## Export the BL and M6 datasets as .csv

In [9]:
export_to_csv(df_bl, f"{DATA_DIR}/4_Initial_Autoimmune_RA_Only_Transposed_BL.csv")
export_to_csv(df_m6, f"{DATA_DIR}/5_Initial_Autoimmune_RA_Only_Transposed_M6.csv")

File saved to: C:\Users\barbs\COMPUTER SCIENCE\FH CAMPUS\6_COURSES SS 26\KI Wahlfach Projekt\Source Code_Main_1\RA-Map-EDA\datasets/4_Initial_Autoimmune_RA_Only_Transposed_BL.csv
File saved to: C:\Users\barbs\COMPUTER SCIENCE\FH CAMPUS\6_COURSES SS 26\KI Wahlfach Projekt\Source Code_Main_1\RA-Map-EDA\datasets/5_Initial_Autoimmune_RA_Only_Transposed_M6.csv


# Perform a harmonisation validation - patient overlap, missing patients, duplicate indices, feature alignment, column order consistency, basic structural sanity


In [10]:
report = harmonisation_validator(df_bl, df_m6)

print(list(df_bl.index[:10]))
print(list(df_m6.index[:10]))


===== HARMONISATION REPORT =====
BL shape: (265, 163)
M6 shape: (235, 163)

Shared patients: 234
BL-only patients: 31
M6-only patients: 1

Feature mismatch BL-only: 0
Feature mismatch M6-only: 0

Feature order identical: True

['TAC1241_BL', 'TAC1147_BL', 'TAC1094_BL', 'TAC1272_BL', 'TAC1096_BL', 'TAC1041_BL', 'TAC1107_BL', 'TAC1186_BL', 'TAC1090_BL', 'TAC1061_BL']
['TAC1241_M6', 'TAC1147_M6', 'TAC1272_M6', 'TAC1096_M6', 'TAC1041_M6', 'TAC1107_M6', 'TAC1186_M6', 'TAC1090_M6', 'TAC1061_M6', 'TAC1019_M6']


## Perform futher checks to see if additional harmonisation steps are needed

In [11]:
# Check variance issues
df_bl.nunique().sort_values().head(10)

1066528482_BCAP31       111
105481284_RPLP2         132
105489246_MS4A1         136
1066559611_CXCL5        138
1066859121_CALR         141
1043138774_MBP          141
1066564363_ZNF217       143
1066859023_LYZ          145
1066861712_CENPB        146
HN1L_1043144040_HN1L    153
dtype: int64

In [12]:
# Check for non-numeric values
df_bl.select_dtypes(include="object").columns

Index([], dtype='str')

In [13]:
# Check zero variance proteins
zero_var = df_bl.columns[df_bl.nunique() <= 1]
print(len(zero_var))

0


In [14]:
# Confirm final shape stability
print(df_bl.shape)

(265, 163)


In [15]:
# Confirm index correctness
df_bl.index.is_unique

True

In [16]:
# Check missing values one last time
df_bl.isna().sum().sum()

np.int64(0)

In [17]:
# Check extreme outliers
df_bl.describe().T[["min","max"]].head()

,min,max
104740305_XRCC6,19.0,5157.0
104741891_SNRPD1,33.0,2984.0
104742995_ACTB,54.0,16363.0
104743232_PTBP1,37.0,22011.0
104743420_FEN1,14.0,2122.5
